In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [12]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [13]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
batch_size = 100
# batch_size = None

# opt = torch_numopt.NewtonRaphson(params=model.parameters(), model=model, lr=0.01, line_search_method="const", batch_size=batch_size)
opt = torch_numopt.NewtonRaphson(params=model.parameters(), model=model, lr=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo", batch_size=batch_size)
# opt = torch_numopt.NewtonRaphson(params=model.parameters(), model=model, lr=1, c1=1e-4, tau=0.5, line_search_method="backtrack", line_search_cond="wolfe", batch_size=batch_size)
# opt = torch_numopt.NewtonRaphson(params=model.parameters(), model=model, lr=1, c1=1e-4, tau=0.5, line_search_method="backtrack", line_search_cond="strong-wolfe", batch_size=batch_size)
# opt = torch_numopt.NewtonRaphson(params=model.parameters(), model=model, lr=1, c1=1e-4, tau=0.5, line_search_method="backtrack", line_search_cond="goldstein", batch_size=batch_size)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.1477172076702118
epoch:  1, loss: 0.13845902681350708
epoch:  2, loss: 0.1364526003599167
epoch:  3, loss: 0.10920850187540054
epoch:  4, loss: 0.08545127511024475
epoch:  5, loss: 0.06542014330625534
epoch:  6, loss: 0.06454538553953171
epoch:  7, loss: 0.056960564106702805
epoch:  8, loss: 0.05058211088180542
epoch:  9, loss: 0.026860257610678673
epoch:  10, loss: 0.010675712488591671
epoch:  11, loss: 0.008533380925655365
epoch:  12, loss: 0.007083031814545393
epoch:  13, loss: 0.005983076058328152
epoch:  14, loss: 0.0052510653622448444
epoch:  15, loss: 0.0047559854574501514
epoch:  16, loss: 0.004410560708492994
epoch:  17, loss: 0.004069101996719837
epoch:  18, loss: 0.0037803323939442635
epoch:  19, loss: 0.003550390712916851
epoch:  20, loss: 0.0033465891610831022
epoch:  21, loss: 0.0031819280702620745
epoch:  22, loss: 0.0030221473425626755
epoch:  23, loss: 0.002894432982429862
epoch:  24, loss: 0.0027686869725584984
epoch:  25, loss: 0.002661413280293345

In [14]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9873911869446659
Test metrics:  R2 = 0.9293441710638282
